In [1]:
from dotenv import load_dotenv
import requests
import json
from datetime import datetime
import use_case.Stock_Price_Prediction.predict.sentiment_predict as sentiment
from itertools import repeat
import pandas as pd
import time
import os
from pathlib import Path

# Get Access Key

In [2]:
load_dotenv();

In [3]:
api_key1 = os.getenv("ALPHA_VANTAGE1")
api_key2 = os.getenv("ALPHA_VANTAGE2")

# Prepare paramters to get news

In [4]:
base_url = f"https://www.alphavantage.co/query?limit=1000&sort=RELEVANCE&function=NEWS_SENTIMENT"
parameters = {
    "topics"   : ['earnings', 'financial_markets' , 'economy_fiscal', 'economy_monetary' , 'economy_macro', 'energy_transportation', 'finance'] ,   
    "ticker"   : "VOO",
    "time_from": "20250101T0000",
    "time_to"  : "20251230T2359",   
}


# Move any list-valued items to the end (keep original order within each group)
parameters = {
        **{k: v for k, v in parameters.items() if not isinstance(v, list)},
        **{k: v for k, v in parameters.items() if isinstance(v, list)},
}

In [5]:
standard = [f"{k}={v}" for k, v in parameters.items() if not isinstance(v, list)]
expanded = [(f"{k}={item}" , item ) for k, v in parameters.items() if isinstance(v, list) for item in v]

full_param_list = [("&".join([*standard, e]) ,t) for e, t in expanded]

# Function to get News

In [6]:
def get_news(parameter):
    
    parameter = '&' + parameter  if parameter != '' and parameter[0] != '&' else parameter         
    
    url = f'{base_url}&apikey={api_key}{parameter}'

    try:
        
        r = requests.get(url)
        data = r.json()   
    
        stock_news = [(item['time_published'][:8], item['summary'].strip()) for item in data['feed'] ]
    
        return stock_news
    

    except Exception as e:
       
        print(e)
        print(data)

# Get News List by topic

In [7]:
def get_full_news_list():

    full_news_list = []
    
    for item, topic in full_param_list:
    
        time.sleep(0.5)
        
        news = get_news( item )
    
        
        if not news == None:
    
            full_news_list.append({"topic": topic,"news" : news })

    return full_news_list

# Call API and Save to txt file

In [ ]:
# full_news_list = get_full_news_list()

# def save_news_txt(name, data):

#     with open(name , 'w') as f:
        
#         json.dump(data, f)    

# save_news_txt('news.txt', full_news_list)

In [ ]:
# for item in full_news_list:
    
#     topic = item['topic']
#     newsList = item['news']

#     print('='*50,'\n', topic, ' :')
#     try:
#         for news in newsList[:2]:
#             date = news[0]
#             sentense = news[1]
    
#             print( '-'*5,date, '-'*40)
#             print(sentense)   
    
#     except Exception as e:
#         print(e)

# Get Data from txt file

In [8]:
with open('news.txt', 'r') as file:
    
    data = json.load(file)

FileNotFoundError: [Errno 2] No such file or directory: 'news.txt'

# Apply Sentiment Analysis to news

In [9]:
predict_result_list = [] 

for item in data:
    
    topic = item['topic']
    newsList = item['news']
    
    dateList = [n[0] for n in newsList]
    sentenseList = [n[1] for n in newsList]

    predicts_list,label_map, result,correct,incorrect = sentiment(sentenseList)    
    result = list(zip(dateList,sentenseList, predicts_list, repeat(topic)))
    
    result = [{"date":d,"topic":t,"sentense":s,"predict":p} for d,s,p,t in result]

    predict_result_list.extend(result)

NameError: name 'data' is not defined

# Organize data and group by Topic + Date

In [ ]:
t_List = [item['topic'] for item in data]
p_List = [v for k,v in label_map.items()]


full_column_list = [f"{t}_{p}" for t in t_List for p in p_List]

In [ ]:
df = pd.DataFrame(predict_result_list)

df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

In [ ]:
pt = df.pivot_table(
    index='date',
    columns=['topic', 'predict'],
    aggfunc='size',
    fill_value=0
)

# flatten MultiIndex columns -> "topic_predict"
pt.columns = [f"{topic}_{pred}" for topic, pred in pt.columns]
pt = pt.reset_index()


In [ ]:
for col in full_column_list:
    if col not in pt.columns:
        pt[col] = 0 


# Reorder columns to match full_column_list (optional)
pt = pt[['date'] + full_column_list]

In [ ]:
pt.to_csv('data/data_alphavantage', index = False)

# Print row count of each month

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

monthly_count = (
    df.dropna(subset=["date"])
      .groupby([df['topic'], df["date"].dt.to_period("M")])
      .size()
      .reset_index(name="row_count")
)

monthly_count["date"] = monthly_count["date"].astype(str)

# === Others ======

In [8]:
topics = parameters['topics']
api_list = [api_key1 , api_key2]
year = 2025


count = 1
split = 1

full_split_list = []
batch_list = [] 

for m in range(12):
    
    from_month = m + 1
    to_month = 12 if from_month == 12 else from_month + 1
    to_day = 30 if from_month == 12 else 1
    
    from_date = f'{year}{str(from_month).zfill(2)}01T0000'
    to_date = f'{year}{str(to_month).zfill(2)}{str(to_day).zfill(2)}T2359'
   
    for topic in topics:
        
        api_idx = count % 2
        api = api_list[api_idx]        

        parameter = f"&ticker=VOO&time_from={from_date}&time_to={to_date}&topics={topic}"
        url = f"{base_url}&apikey={api}{parameter}"    
        
        file_name = f'news_{year}{str(from_month).zfill(2)}_{year}{str(to_month).zfill(2)}{str(to_day).zfill(2)}_{topic}.txt'
        
        # print(f'{str(count).zfill(2)}) {url}')

        batch_list.append( ( file_name,  url ,topic) )

        if split % 50 == 0:

            full_split_list.append(batch_list)
            batch_list = []
            
            # print("--"* 80)
            split = 1

        count += 1
        split += 1 

    
if len(batch_list) > 0 :
    
    full_split_list.append(batch_list)

In [21]:
def handle_batch( call_list ):
    
    for call_item in call_list:

        file_name = f'data/{call_item[0]}'
        url = call_item[1]
        topic = call_item[2]
            
        if not Path(file_name).exists():

            try:
                
                # Call API
                r = requests.get(url)
                data = r.json() 
                
                batch_news_list = [(item['time_published'][:8], item['summary'].strip()) for item in data['feed'] ]
                batch_news_list = {"topic": topic,"news" : batch_news_list }
                
                # Save to txt
                save_news_txt(file_name, batch_news_list)
    
                print(f'Saved {call_item[0]}')
                
            except Exception as e:

                print(f'Failed {call_item[0]}' )
                # print(data['Information'].strip())
                print('-' * 60)

## Day 1

In [ ]:
handle_batch( full_split_list[0] )

## Day 2

In [ ]:
handle_batch( full_split_list[1] )

## Union all txt

In [17]:
import re

full_news = []

reg = re.compile(r'news_[0-9]+_[0-9]+_.+\.txt')


for file in os.listdir('data/'):
    file_path = f'data/{file}'
    
    if reg.match(file):
        
        with open(file_path, 'r') as file:
    
            data = json.load(file)

            full_news.append(data)



In [19]:
predict_result_list = [] 

for item in full_news:
    
    topic = item['topic']
    newsList = item['news']
    
    dateList = [n[0] for n in newsList]
    sentenseList = [n[1] for n in newsList]

    predicts_list,label_map, result,correct,incorrect = sentiment(sentenseList)    
    result = list(zip(dateList,sentenseList, predicts_list, repeat(topic)))
    
    result = [{"date":d,"topic":t,"sentense":s,"predict":p} for d,s,p,t in result]

    predict_result_list.extend(result)

In [21]:
predict_result_list

[{'date': '20250201',
  'topic': 'earnings',
  'sentense': 'First Trust Exchange-Traded AlphaDEX Fund - First Trust Energy AlphaDEX Fund announced a Quarterly dividend of USD 0.1141 per share, payable on December 31, 2025. The ex-date and record date for this dividend are both December 12, 2025. This information was published on February 1, 2025, by S&P Capital IQ.',
  'predict': 2},
 {'date': '20250201',
  'topic': 'earnings',
  'sentense': "Li Auto reported delivering 29,927 vehicles in January 2025, bringing cumulative deliveries to 1,163,799 vehicles. The Li L6 model reached 200,000 cumulative deliveries and maintained its position as China's best-selling extended-range electric vehicle for seven consecutive months. The company continues to lead the premium Chinese automotive market and has significantly expanded its retail and charging infrastructure, while also enhancing autonomous driving capabilities with the new OTA update 7.0.",
  'predict': 2},
 {'date': '20250201',
  'topic